In [ ]:
"""
=============================================================
 #NOTEBOOK 3 — Membrane Volume Feature Engineering
=============================================================
"""This notebook implements a novel feature engineering approach
applied to the pH/temperature chill curve time-series data 
collected at hourly intervals post-slaughter.

Concept:
  At each timepoint, pH × temperature gives the instantaneous 
  'height' of a 3D surface (time × temperature × pH). 
  Integrating that product over time yields a single value per 
  carcass — the volume under the membrane — which proxies 
  cumulative proteolytic enzyme activity during the chill period,
  a key driver of meat tenderness.

Features engineered:
  - membrane_vol_full:  volume under full pH×temp×time surface
  - membrane_vol_early: volume over the early chill window
  - membrane_vol_late:  volume over the late chill window
  - pH_AUC_to_Xhr:      cumulative pH AUC at each timepoint
  - QC flags:           missingness counts and interpolation flags

Missingness strategy:
  - Interior gaps: linearly interpolated across the time axis
  - Edge gaps: not extrapolated, integral computed over 
    available contiguous window
  - Fully missing rows: volume feature set to NaN

Outputs:
  - ALL_LD_carcass_level_with_membrane.csv  (Loin)
  - ALL_SM_carcass_level_with_membrane.csv  (Topside)"""
=============================================================
"""

In [ ]:
# Evaluation of APB Tem & PH time series data on target
# Trial of  CatBoostRegp learning model
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pandas.plotting import scatter_matrix
import seaborn as sns
import scipy.stats as stats
import statsmodels.api as sm
import csv
from scipy.stats import chi2_contingency
import matplotlib.pyplot as plt

from openpyxl import Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.styles import Font, Alignment
import re
import textwrap
from statsmodels.stats.contingency_tables import mcnemar
import sqlite3
from contextlib import contextmanager
import json
import csv
from fuzzywuzzy import process
from itertools import product
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor, Pool
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
import mlflow
from catboost import CatBoostRegressor, Pool

In [ ]:
"""
membrane_volume_features.py
----------------------------
Computes "volume under the pH-temperature-time membrane" features
for the Loin (LD) and Topside (SM) carcass-level datasets.

Feature concept:
    At each timepoint, pH × temperature gives the instantaneous "height"
    of a 3D surface (time × temp × pH). Integrating that product over time
    gives a single number per carcass — the volume under the membrane —
    which proxies cumulative proteolytic enzyme activity during the chill.

Missingness strategy:
    - Interior gaps: linearly interpolated across the time axis (valid
      because pH decline and temperature drop are smooth, monotonic processes)
    - Edge gaps (missing first or last reading): not extrapolated — the
      integral is computed over whatever contiguous window is available
    - Rows missing ALL timepoints: volume feature set to NaN
    - A `n_missing_ph` and `n_missing_temp` column is added per cut as a
      QC flag (can be used as a CatBoost feature or filtered on)

Outputs:
    ALL_LD_carcass_level_with_membrane.csv
    ALL_SM_carcass_level_with_membrane.csv
"""

import numpy as np
import pandas as pd

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

CUTS = {
    "Loin": {
        "input_path": "/ALL_LD_carcass_level.csv",
        "output_path": "/ALL_LD_carcass_level_with_membrane.csv",
        "hours": [1, 2, 4, 6, 9, 14, 19, 24],
        "ph_cols": [
            "1hr - Loin - pH",
            "2hr - Loin  - pH",
            "4hr - Loin  - pH",
            "6hr - Loin  - pH",
            "9hr - Loin  - pH",
            "14hr - Loin  - pH",
            "19hr - Loin  - pH",
            "24hr - Loin  - pH",
        ],
        "temp_cols": [
            "1hr - Loin - Temp",
            "2hr - Loin  - Temp",
            "4hr - Loin  - Temp",
            "6hr - Loin  - Temp",
            "9hr - Loin  - Temp",
            "14hr - Loin  - Temp",
            "19hr - Loin  - Temp",
            "24hr - Loin  - Temp",
        ],
    },
    "Topside": {
        "input_path": "/ALL_SM_carcass_level.csv",
        "output_path": "/ALL_SM_carcass_level_with_membrane.csv",
        # 1hr is 100% missing for Topside — excluded from integration window
        "hours": [2, 4, 6, 9, 14, 19, 24],
        "ph_cols": [
            "2hr - Topside - pH",
            "4hr - Topside - pH",
            "6hr - Topside - pH",
            "9hr - Topside - pH",
            "14hr - Topside - pH",
            "19hr - Topside - pH",
            "24hr-Topside - pH",
        ],
        "temp_cols": [
            "2hr - Topside - Temp",
            "4hr - Topside - Temp",
            "6hr - Topside - Temp",
            "9hr - Topside - Temp",
            "14hr - Topside - Temp",
            "19hr - Topside - Temp",
            "24hr-Topside - Temp",
        ],
    },
}


# ---------------------------------------------------------------------------
# Core interpolation helper
# ---------------------------------------------------------------------------

def interpolate_row(values: np.ndarray, hours: list) -> np.ndarray:
    """
    Linearly interpolate interior NaNs in a single row of time-series values.
    Edge NaNs (leading / trailing) are left as NaN — no extrapolation.

    Parameters
    ----------
    values : np.ndarray  shape (n_timepoints,)
    hours  : list        corresponding time axis values

    Returns
    -------
    np.ndarray  same shape, interior NaNs filled, edges unchanged
    """
    s = pd.Series(values, index=hours, dtype=float)
    # interpolate(method='index') uses the actual numeric index (hours),
    # giving true linear interpolation in time rather than equal-spaced.
    # limit_direction='both' would extrapolate — we use 'forward' only
    # then 'backward' only to fill interior gaps without touching edges.
    s_interp = s.interpolate(method="index", limit_direction="forward")
    s_interp = s_interp.interpolate(method="index", limit_direction="backward")
    # Edges that were NaN remain NaN because we never extrapolate
    return s_interp.values


# ---------------------------------------------------------------------------
# Volume feature computation
# ---------------------------------------------------------------------------

def compute_membrane_features(df: pd.DataFrame, cfg: dict) -> pd.DataFrame:
    """
    Add membrane volume features to df in-place and return it.

    Features added
    --------------
    ph_x_temp_Xhr        : pH × temp product at each timepoint (post-interpolation)
    membrane_vol_full    : volume under the full pH×temp×time surface (trapz)
    membrane_vol_early   : volume over the early window (first half of timepoints)
    membrane_vol_late    : volume over the late window (second half of timepoints)
    n_missing_ph         : number of missing pH readings before interpolation
    n_missing_temp       : number of missing temp readings before interpolation
    membrane_interp_flag : True if any interpolation was applied to this row
    """
    hours     = cfg["hours"]
    ph_cols   = cfg["ph_cols"]
    temp_cols = cfg["temp_cols"]
    n_tp      = len(hours)

    # -- Record raw missingness counts before any interpolation ---------------
    df["n_missing_ph"]   = df[ph_cols].isna().sum(axis=1)
    df["n_missing_temp"] = df[temp_cols].isna().sum(axis=1)
    df["membrane_interp_flag"] = (df["n_missing_ph"] > 0) | (df["n_missing_temp"] > 0)

    # -- Extract arrays -------------------------------------------------------
    ph_arr   = df[ph_cols].values.astype(float)    # shape (n_rows, n_tp)
    temp_arr = df[temp_cols].values.astype(float)

    # -- Interpolate row-by-row -----------------------------------------------
    ph_interp   = np.array([interpolate_row(ph_arr[i],   hours) for i in range(len(df))])
    temp_interp = np.array([interpolate_row(temp_arr[i], hours) for i in range(len(df))])

    # -- Store interpolated point-wise product --------------------------------
    product = ph_interp * temp_interp   # shape (n_rows, n_tp)

    for j, hr in enumerate(hours):
        df[f"ph_x_temp_{hr}hr"] = product[:, j]

    # -- Compute volumes via np.trapz -----------------------------------------
    # Full window
    df["membrane_vol_full"] = _trapz_with_nans(product, hours)

    # Early window: first ceil(n/2) timepoints
    mid = int(np.ceil(n_tp / 2))
    df["membrane_vol_early"] = _trapz_with_nans(product[:, :mid], hours[:mid])

    # Late window: last floor(n/2) timepoints
    df["membrane_vol_late"] = _trapz_with_nans(product[:, mid:], hours[mid:])

    # -- Cumulative AUC at each timepoint (pure pH, for comparison) -----------
    for i in range(2, n_tp + 1):
        feat = f"pH_AUC_to_{hours[i-1]}hr"
        df[feat] = _trapz_with_nans(ph_interp[:, :i], hours[:i])

    return df


def _trapz_with_nans(arr: np.ndarray, hours: list) -> np.ndarray:
    """
    Row-wise trapezoidal integration, skipping rows where all values are NaN.
    Rows with some NaN (edge cases post-interpolation) integrate over the
    available contiguous non-NaN window.

    Parameters
    ----------
    arr   : np.ndarray  shape (n_rows, n_timepoints)
    hours : list        time axis

    Returns
    -------
    np.ndarray  shape (n_rows,)
    """
    hours_arr = np.array(hours, dtype=float)
    results = np.full(arr.shape[0], np.nan)

    for i, row in enumerate(arr):
        valid = ~np.isnan(row)
        if valid.sum() < 2:
            # Need at least 2 points to integrate
            continue
        results[i] = np.trapezoid(row[valid], hours_arr[valid])

    return results


# ---------------------------------------------------------------------------
# Summary / QC report
# ---------------------------------------------------------------------------

def print_qc_report(df: pd.DataFrame, cut_name: str) -> None:
    print(f"\n{'='*60}")
    print(f"  QC Report — {cut_name}  (n={len(df)})")
    print(f"{'='*60}")

    interp_rows = df["membrane_interp_flag"].sum()
    print(f"  Rows with interpolation applied : {interp_rows} ({interp_rows/len(df)*100:.1f}%)")

    null_vol = df["membrane_vol_full"].isna().sum()
    print(f"  Rows with NaN membrane_vol_full : {null_vol} ({null_vol/len(df)*100:.1f}%)")

    print(f"\n  membrane_vol_full  stats:")
    print(df["membrane_vol_full"].describe().to_string())

    print(f"\n  membrane_vol_early stats:")
    print(df["membrane_vol_early"].describe().to_string())

    print(f"\n  membrane_vol_late  stats:")
    print(df["membrane_vol_late"].describe().to_string())

    print(f"\n  Missing timepoint distribution (pH):")
    vc = df["n_missing_ph"].value_counts().sort_index()
    for n, cnt in vc.items():
        label = "(interpolated)" if n > 0 else "(complete)"
        print(f"    {n} missing: {cnt:>4} rows  {label}")


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

def main():
    for cut_name, cfg in CUTS.items():
        print(f"\nProcessing {cut_name}...")
        df = pd.read_csv(cfg["input_path"])
        df = compute_membrane_features(df, cfg)
        print_qc_report(df, cut_name)
        df.to_csv(cfg["output_path"], index=False)
        print(f"  Saved -> {cfg['output_path']}")

    print("\nDone.")


if __name__ == "__main__":
    main()